In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

In [18]:
import os
import json
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import faiss
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

In [62]:
sample_farmer_data = [
    {
        "farmer_id": "FARMER001",
        "full_name": "Nguyễn Văn An",
        "farm_name": "Ruộng An Phát",
        "area_ha": 2.5,
        "planting_date": "2025-09-20",
        "days_since_planting": 44,             
        "rice_variety": "ST25", 
        "location": {
            "province": "An Giang"
        },
        "iot_data": { 
            "temperature": 29.5,
            "humidity": 83,
            "soil_moisture": 65,
            "soil_ph": 5.8,
            "water_level": 10,
            "lux": 48200,
            "wind": 2.8,
            "wind_avg": 2.4,
            "detected_disease_name": "blast",
            "disease_confidence": 0.97
        },
        "summary_3d": {  
            "1/11/2025": {
                "temperature": 28.6, "humidity": 91, "soil_moisture": 72, "soil_ph": 5.9,
                "water_level": 15, "lux": 35800, "wind": 1.3, "wind_avg": 1.1
            },
            "2/11/2025": {
                "temperature": 28.9, "humidity": 88, "soil_moisture": 69, "soil_ph": 5.8,
                "water_level": 13, "lux": 41200, "wind": 1.7, "wind_avg": 1.5
            },
            "3/11/2025": {  
                "temperature": 29.5, "humidity": 83, "soil_moisture": 65, "soil_ph": 5.8,
                "water_level": 10, "lux": 48200, "wind": 2.8, "wind_avg": 2.4
            }
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-01T00:00:00Z",   
            "stage_name": "Bón Thúc 2 - Làm Đòng/Đón Đòng",
            "main_message": "CẦN BÓN THÚC 2 NGAY NGÀY MAI 1/11/2025 (7h sáng) vì đây là ngày đẹp nhất trong 3 ngày tới: mực nước lý tưởng 13cm, độ ẩm đất cao, trời râm mát, gió nhẹ, rất thuận lợi cho cây hấp thụ phân và hạn chế thất thoát.",
            "analysis": {
                "nutrient_need_assessment": "Cây lúa 44 ngày sau sạ đã vào cuối giai đoạn làm đòng. IoT hiện tại cho thấy nước ruộng chỉ còn 10cm (quá cạn), độ ẩm đất giảm xuống 65%, pH 5.8 và đặc biệt phát hiện bệnh đạo ôn nặng (97%). Cần bón thúc 2 ngay để bổ sung NPK + Kali đậm đặc giúp cây bật đòng mạnh và tăng sức chống chịu bệnh (1)(2).",
                "optimal_timing_summary": "Trong 3 ngày tới, ngày 1/11/2025 là ngày tối ưu nhất: mực nước 15cm (lý tưởng để bón phân), độ ẩm đất 91%, nhiệt độ 28.6°C, trời nhiều mây, gió nhẹ <2km/h, xác suất mưa rất thấp → điều kiện hoàn hảo để bón phân và cây hấp thụ tốt nhất."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "40-45 ngày sau sạ",
                    "objective": "Thúc đẩy làm đòng mạnh, đón đòng đều, tăng số bông chắc, nâng cao khả năng chống chịu bệnh đạo ôn đang bùng phát",
                    "key_indicators": "Cây bắt đầu 'đi đòng' (đòng nhú trong bẹ lá), lá đứng, bẹ lá vàng nhạt. IoT phát hiện blast 97% → cần tăng Kali và Silic gấp để tăng sức đề kháng (1)(3).",
                    "fertilizers": [
                        {
                            "type": "NPK 20-20-15 + TE (hoặc Đầu Trâu Thúc Đòng)",
                            "recommended_dosage_per_unit": "220-260 kg/ha (liều cao cho ruộng nhiễm đạo ôn) (1)",
                            "calculation_details": "2.5 ha × 240 kg/ha = 600 kg",
                            "quantity_kg": 600.0,
                            "instructions": "Bón vãi đều vào sáng sớm ngày 2/11 khi mặt ruộng se khô, giữ mực nước 4-6 cm trong 4-5 ngày sau bón."
                        },
                        {
                            "type": "Kali Clorua (KCl 60%)",
                            "recommended_dosage_per_unit": "60-80 kg/ha (bổ sung riêng chống đạo ôn) (2)",
                            "calculation_details": "2.5 ha × 70 kg/ha = 175 kg",
                            "quantity_kg": 175.0,
                            "instructions": "Trộn chung với NPK hoặc bón riêng, ưu tiên buổi sáng."
                        },
                        {
                            "type": "Phân bón lá Silic + vi lượng (SiO2 ≥ 15%)",
                            "recommended_dosage_per_unit": "1.2-1.5 lít/ha, pha 600-800 lần (3)",
                            "calculation_details": "2.5 ha × 1.3 lít/ha = 3.25 lít (làm tròn 3.3 lít)",
                            "quantity_kg": 3.3,
                            "instructions": "Phun đồng thời với thuốc BVTV đặc trị đạo ôn vào chiều ngày 2/11 hoặc sáng ngày 3/11."
                        }
                    ],
                    "important_notes": "Cảnh báo: Ruộng đang nhiễm đạo ôn nặng → tuyệt đối không bón thừa đạm. Sau bón phải giữ nước liên tục 5-7 ngày, không để khô hạn. Nếu mưa lớn ngày 2/11 thì hoãn sang sáng ngày 3/11 và bổ sung thêm 20% Kali."
                }
            ],
            "next_key_stage": "Theo dõi giai đoạn trổ bông (khoảng 55-65 ngày sau sạ) → chuẩn bị bón đón đòng muộn hoặc bón lá nuôi hạt nếu cây yếu"
        }
    },
    {
        "farmer_id": "FARMER002",
        "full_name": "Trần Thị Bé",
        "farm_name": "Ruộng Bé Hương",
        "area_ha": 1.8,
        "planting_date": "2025-08-15",
        "days_since_planting": 80,
        "rice_variety": "OM18",
        "location": {
            "province": "Đồng Tháp"
        },
        "iot_data": {
            "temperature": 31.2,
            "humidity": 78,
            "soil_moisture": 58,
            "soil_ph": 6.1,
            "water_level": 8,
            "lux": 68200,
            "wind": 4.1,
            "wind_avg": 3.7,
            "detected_disease_name": "none",
            "disease_confidence": 0.12
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 30.8, "humidity": 82, "soil_moisture": 62, "soil_ph": 6.0, "water_level": 12, "lux": 59200, "wind": 2.9, "wind_avg": 2.5},
            "2/11/2025": {"temperature": 31.5, "humidity": 76, "soil_moisture": 55, "soil_ph": 6.1, "water_level": 7,  "lux": 71200, "wind": 4.5, "wind_avg": 4.0},
            "3/11/2025": {"temperature": 31.2, "humidity": 78, "soil_moisture": 58, "soil_ph": 6.1, "water_level": 8,  "lux": 68200, "wind": 4.1, "wind_avg": 3.7}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-03T00:00:00Z",
            "stage_name": "Bón Đón Đòng Muộn & Nuôi Hạt",
            "main_message": "CẦN BÓN ĐÓN ĐÒNG MUỘN + PHUN LÁ NGAY SÁNG 3/11/2025 vì cây đã 80 ngày, đang giai đoạn trổ đều, thời tiết 3/11 nắng nhẹ, gió mạnh vừa phải, rất thuận lợi cho phun thuốc và bón phân lá.",
            "analysis": {
                "nutrient_need_assessment": "Cây lúa OM18 80 ngày tuổi đang trổ đều → cần bổ sung Kali và vi lượng gấp để nuôi hạt, hạn chế lép hạt do thời tiết nắng nóng gần đây. Mực nước hiện chỉ 8cm, đất hơi khô (58%), cần tăng cường phân bón lá và giữ nước tốt hơn.",
                "optimal_timing_summary": "Ngày 3/11 là ngày đẹp nhất trong 3 ngày tới: nắng đều, gió 3-4m/s giúp thuốc bám tốt, nhiệt độ 31.2°C phù hợp để phun buổi sáng sớm hoặc chiều mát."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "75-85 ngày sau sạ (trổ đều)",
                    "objective": "Nuôi hạt chắc, hạn chế lép, tăng tỷ lệ hạt chắc và khối lượng nghìn hạt",
                    "key_indicators": "Bông đã trổ đều >70%, hạt bắt đầu vào chắc xanh, lá đốt cuối còn xanh nhưng hơi vàng chân lá do thiếu Kali.",
                    "fertilizers": [
                        {
                            "type": "NPK 15-15-20 + TE (hoặc Đầu Trâu Nuôi Hạt)",
                            "recommended_dosage_per_unit": "120-150 kg/ha",
                            "calculation_details": "1.8 ha × 140 kg/ha = 252 kg",
                            "quantity_kg": 252.0,
                            "instructions": "Bón vãi đều khi mặt ruộng se khô, sau đó tháo nước vào giữ 3-5cm."
                        },
                        {
                            "type": "Phân bón lá cao Kali + Bo + Zn",
                            "recommended_dosage_per_unit": "1.5-2 kg/ha, pha 600-800 lần",
                            "calculation_details": "1.8 ha × 1.8 kg/ha = 3.24 kg (làm tròn 3.3 kg)",
                            "quantity_kg": 3.3,
                            "instructions": "Phun 2 lần cách nhau 5-7 ngày, lần 1 vào sáng 3/11."
                        }
                    ],
                    "important_notes": "Giữ nước liên tục 3-5cm đến khi thu hoạch, tuyệt đối tránh để khô nước giai đoạn vào chắc. Nếu nhiệt độ >34°C thì phun vào 6-8h sáng hoặc sau 16h."
                }
            ],
            "next_key_stage": "Chuẩn bị thu hoạch (dự kiến 92-98 ngày sau sạ) → theo dõi độ chín vàng, cắt nước đúng thời điểm"
        }
    },
    {
        "farmer_id": "FARMER003",
        "full_name": "Lê Văn Hùng",
        "farm_name": "Ruộng Hùng Cường",
        "area_ha": 3.2,
        "planting_date": "2025-10-01",
        "days_since_planting": 33,
        "rice_variety": "Jasmine 85",
        "location": {
            "province": "Kiên Giang"
        },
        "iot_data": {
            "temperature": 27.8,
            "humidity": 92,
            "soil_moisture": 78,
            "soil_ph": 5.5,
            "water_level": 18,
            "lux": 31200,
            "wind": 1.2,
            "wind_avg": 0.9,
            "detected_disease_name": "brown_spot",
            "disease_confidence": 0.89
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 27.5, "humidity": 94, "soil_moisture": 81, "soil_ph": 5.4, "water_level": 20, "lux": 28500, "wind": 0.8, "wind_avg": 0.7},
            "2/11/2025": {"temperature": 28.1, "humidity": 90, "soil_moisture": 76, "soil_ph": 5.6, "water_level": 16, "lux": 34200, "wind": 1.5, "wind_avg": 1.3},
            "3/11/2025": {"temperature": 27.8, "humidity": 92, "soil_moisture": 78, "soil_ph": 5.5, "water_level": 18, "lux": 31200, "wind": 1.2, "wind_avg": 0.9}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-02T00:00:00Z",
            "stage_name": "Bón Thúc Đón Đòng + Xử lý bệnh đốm nâu",
            "main_message": "CẦN BÓN THÚC ĐÓN ĐÒNG + PHUN THUỐC TRỊ ĐỐM NÂU NGAY NGÀY 2/11/2025 vì cây 33 ngày đang đẻ nhánh rộ, phát hiện bệnh đốm nâu nặng (89%), thời tiết ngày 2/11 rất thuận lợi để phun thuốc.",
            "analysis": {
                "nutrient_need_assessment": "Cây đang giai đoạn đẻ nhánh cuối - chuẩn bị làm đòng, pH đất hơi thấp (5.5), bệnh đốm nâu xuất hiện mạnh do thừa nước và thiếu Kali.",
                "optimal_timing_summary": "Ngày 2/11 trời âm u, độ ẩm cao, gió nhẹ → điều kiện lý tưởng để phun thuốc trừ bệnh đốm nâu và bón phân."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "30-35 ngày sau sạ",
                    "objective": "Kết thúc đẻ nhánh, thúc làm đòng sớm, tăng sức chống bệnh đốm nâu",
                    "key_indicators": "Số nhánh hữu hiệu đạt đỉnh, lá có đốm nâu tròn viền vàng, pH đất thấp.",
                    "fertilizers": [
                        {
                            "type": "NPK 20-15-15 + TE",
                            "recommended_dosage_per_unit": "180-220 kg/ha",
                            "calculation_details": "3.2 ha × 200 kg/ha = 640 kg",
                            "quantity_kg": 640.0,
                            "instructions": "Bón vào sáng sớm, sau đó giữ nước 5-7cm."
                        },
                        {
                            "type": "Thuốc trừ bệnh đốm nâu (Propiconazole + Tricyclazole)",
                            "recommended_dosage_per_unit": "0.6-0.8 lít/ha",
                            "calculation_details": "3.2 ha × 0.7 lít/ha = 2.24 lít (làm tròn 2.3 lít)",
                            "quantity_kg": 2.3,
                            "instructions": "Phun kỹ 2 mặt lá vào chiều ngày 2/11, lặp lại sau 7-10 ngày nếu bệnh còn nặng."
                        }
                    ],
                    "important_notes": "Tháo bớt nước xuống còn 10-12cm trước khi phun thuốc, sau phun 2 ngày mới bón phân."
                }
            ],
            "next_key_stage": "Theo dõi giai đoạn làm đòng (40-50 ngày sau sạ)"
        }
    },
    {
        "farmer_id": "FARMER004",
        "full_name": "Huỳnh Kim Thảo",
        "farm_name": "Ruộng Thảo Nguyên",
        "area_ha": 4.0,
        "planting_date": "2025-09-10",
        "days_since_planting": 54,
        "rice_variety": "RVT",
        "location": {
            "province": "Sóc Trăng"
        },
        "iot_data": {
            "temperature": 28.4,
            "humidity": 85,
            "soil_moisture": 70,
            "soil_ph": 6.0,
            "water_level": 12,
            "lux": 45200,
            "wind": 3.2,
            "wind_avg": 2.8,
            "detected_disease_name": "sheath_blight",
            "disease_confidence": 0.93
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 28.1, "humidity": 88, "soil_moisture": 74, "soil_ph": 6.0, "water_level": 15, "lux": 39800, "wind": 2.1, "wind_avg": 1.9},
            "2/11/2025": {"temperature": 28.7, "humidity": 82, "soil_moisture": 68, "soil_ph": 6.1, "water_level": 10, "lux": 49200, "wind": 3.8, "wind_avg": 3.4},
            "3/11/2025": {"temperature": 28.4, "humidity": 85, "soil_moisture": 70, "soil_ph": 6.0, "water_level": 12, "lux": 45200, "wind": 3.2, "wind_avg": 2.8}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-01T00:00:00Z",
            "stage_name": "Phun thuốc trị lem lép hạt & khô vằn + bón Kali bổ sung",
            "main_message": "KHẨN CẤP: PHUN THUỐC TRỊ KHÔ VẨN + LEM LÉP HẠT NGAY NGÀY 1/11/2025 vì bệnh khô vằn đang bùng phát mạnh (93%) ở giai đoạn trổ lác đác.",
            "analysis": {
                "nutrient_need_assessment": "Cây RVT 54 ngày đang trổ lác đác, bệnh khô vằn (sheath blight) rất nặng do mật độ cao và độ ẩm liên tục cao.",
                "optimal_timing_summary": "Ngày 1/11 có mây nhiều, gió nhẹ, độ ẩm cao → điều kiện cực kỳ tốt để phun thuốc trừ nấm."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "Giai đoạn trổ lác đác - trổ đều",
                    "objective": "Bảo vệ bông, phòng lem lép hạt và khô vằn",
                    "key_indicators": "Vết bệnh khô vằn lan rộng ở bẹ lá dưới, bông bắt đầu ló ra.",
                    "fertilizers": [
                        {
                            "type": "Thuốc trừ nấm Validamycin + Hexaconazole",
                            "recommended_dosage_per_unit": "0.8-1.0 lít/ha",
                            "calculation_details": "4.0 ha × 0.9 lít/ha = 3.6 lít",
                            "quantity_kg": 3.6,
                            "instructions": "Phun ướt đều gốc và thân lá vào chiều 1/11, phun lần 2 sau 7 ngày."
                        },
                        {
                            "type": "Kali Clorua bổ sung",
                            "recommended_dosage_per_unit": "50 kg/ha",
                            "calculation_details": "4.0 ha × 50 kg/ha = 200 kg",
                            "quantity_kg": 200.0,
                            "instructions": "Bón bổ sung sau khi phun thuốc 2-3 ngày."
                        }
                    ],
                    "important_notes": "Giữ nước nông 5-8cm, không để ngập sâu. Tránh phun khi trời mưa."
                }
            ],
            "next_key_stage": "Theo dõi trổ đều → chuẩn bị bón nuôi hạt"
        }
    },
    {
        "farmer_id": "FARMER005",
        "full_name": "Phạm Văn Tâm",
        "farm_name": "Ruộng Tâm Phúc",
        "area_ha": 2.0,
        "planting_date": "2025-09-25",
        "days_since_planting": 39,
        "rice_variety": "Đài Thơm 8",
        "location": {
            "province": "Tiền Giang"
        },
        "iot_data": {
            "temperature": 29.8,
            "humidity": 80,
            "soil_moisture": 62,
            "soil_ph": 5.9,
            "water_level": 9,
            "lux": 55200,
            "wind": 3.5,
            "wind_avg": 3.1,
            "detected_disease_name": "none",
            "disease_confidence": 0.08
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 29.5, "humidity": 84, "soil_moisture": 68, "soil_ph": 5.9, "water_level": 14, "lux": 48200, "wind": 2.4, "wind_avg": 2.1},
            "2/11/2025": {"temperature": 30.1, "humidity": 77, "soil_moisture": 59, "soil_ph": 6.0, "water_level": 8,  "lux": 61200, "wind": 4.2, "wind_avg": 3.8},
            "3/11/2025": {"temperature": 29.8, "humidity": 80, "soil_moisture": 62, "soil_ph": 5.9, "water_level": 9,  "lux": 55200, "wind": 3.5, "wind_avg": 3.1}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-01T00:00:00Z",
            "stage_name": "Bón Thúc 1 - Đẻ Nhánh Rộ",
            "main_message": "CẦN BÓN THÚC 1 NGAY NGÀY 1/11/2025 (sáng sớm) vì cây 39 ngày đang đẻ nhánh tối đa, dự báo 1/11 có mực nước lý tưởng 14cm, trời râm, rất thuận lợi bón phân.",
            "analysis": {
                "nutrient_need_assessment": "Cây đang giai đoạn đẻ nhánh mạnh nhất, cần bổ sung đạm + lân cao để đạt mật độ nhánh tối ưu.",
                "optimal_timing_summary": "Ngày 1/11 là ngày đẹp nhất: mực nước 14cm, độ ẩm đất cao, trời nhiều mây → cây hấp thụ tốt, ít thất thoát phân."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "35-40 ngày sau sạ",
                    "objective": "Tăng số nhánh hữu hiệu, tạo tiền đề cho năng suất cao",
                    "key_indicators": "Cây đang đẻ nhánh rộ, lá xanh đậm, chưa có dấu hiệu già.",
                    "fertilizers": [
                        {
                            "type": "Urea (46% N)",
                            "recommended_dosage_per_unit": "90-110 kg/ha",
                            "calculation_details": "2.0 ha × 100 kg/ha = 200 kg",
                            "quantity_kg": 200.0,
                            "instructions": "Bón vãi đều khi mặt ruộng se khô, sau bón giữ nước 3-5cm trong 4 ngày."
                        },
                        {
                            "type": "DAP (18-46-0)",
                            "recommended_dosage_per_unit": "80-100 kg/ha",
                            "calculation_details": "2.0 ha × 90 kg/ha = 180 kg",
                            "quantity_kg": 180.0,
                            "instructions": "Trộn chung với Urea hoặc bón riêng."
                        }
                    ],
                    "important_notes": "Không bón khi trời mưa to. Nếu sau 3 ngày cây vẫn vàng lá thì bổ sung thêm 20kg Urea/ha."
                }
            ],
            "next_key_stage": "Theo dõi kết thúc đẻ nhánh (45-50 ngày) → chuẩn bị bón đón đòng"
        }
    },
    {
        "farmer_id": "FARMER007",
        "full_name": "Trương Quốc Bảo",
        "farm_name": "Ruộng Bảo Lộc",
        "area_ha": 3.5,
        "planting_date": "2025-09-05",
        "days_since_planting": 57,
        "rice_variety": "OM5451",
        "location": {"province": "Cần Thơ"},
        "iot_data": {
            "temperature": 28.1, "humidity": 87, "soil_moisture": 71, "soil_ph": 5.7,
            "water_level": 11, "lux": 42800, "wind": 2.9, "wind_avg": 2.5,
            "detected_disease_name": "blast", "disease_confidence": 0.94
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 27.8, "humidity": 90, "soil_moisture": 75, "soil_ph": 5.7, "water_level": 14, "lux": 37500, "wind": 1.8, "wind_avg": 1.6},
            "2/11/2025": {"temperature": 28.5, "humidity": 84, "soil_moisture": 68, "soil_ph": 5.8, "water_level": 9, "lux": 46800, "wind": 3.4, "wind_avg": 3.0},
            "3/11/2025": {"temperature": 28.1, "humidity": 87, "soil_moisture": 71, "soil_ph": 5.7, "water_level": 11, "lux": 42800, "wind": 2.9, "wind_avg": 2.5}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-01T00:00:00Z",
            "stage_name": "Phun thuốc đặc trị đạo ôn cổ bông + bón Kali cứu cây",
            "main_message": "KHẨN CẤP: PHUN THUỐC TRỊ ĐẠO ÔN CỔ BÔNG NGAY SÁNG 1/11/2025 vì bệnh đạo ôn đang bùng phát cực mạnh (94%) ở giai đoạn trổ lác đác, đồng thời bón Kali tăng sức đề kháng.",
            "analysis": {
                "nutrient_need_assessment": "Cây OM5451 57 ngày đang trổ lác đác, đạo ôn cổ bông xuất hiện nặng do trời âm u liên tục và thiếu Kali nghiêm trọng.",
                "optimal_timing_summary": "Ngày 1/11 trời nhiều mây, gió nhẹ, độ ẩm cao → điều kiện lý tưởng để thuốc trừ nấm phát huy tối đa hiệu lực."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "Trổ lác đác đến trổ đều",
                    "objective": "Bảo vệ cổ bông, ngăn đạo ôn lây lan, cứu tỷ lệ hạt chắc",
                    "key_indicators": "Vết đạo ôn trắng xám trên cổ bông và lá đòn, tỷ lệ bệnh >20% khóm.",
                    "fertilizers": [
                        {
                            "type": "Tricyclazole 75WP hoặc Kasumin + Validamycin",
                            "recommended_dosage_per_unit": "0.6-0.8 kg/ha",
                            "calculation_details": "3.5 ha × 0.7 kg/ha = 2.45 kg (làm tròn 2.5 kg)",
                            "quantity_kg": 2.5,
                            "instructions": "Phun ướt đều cổ bông và lá đòn vào 6-8h sáng 1/11, phun lần 2 sau 5-7 ngày."
                        },
                        {
                            "type": "Kali Clorua (KCl 60%) bổ sung khẩn cấp",
                            "recommended_dosage_per_unit": "80-100 kg/ha",
                            "calculation_details": "3.5 ha × 90 kg/ha = 315 kg",
                            "quantity_kg": 315.0,
                            "instructions": "Bón ngay sau phun thuốc 2 ngày, giữ nước 5-7cm."
                        }
                    ],
                    "important_notes": "Tuyệt đối không bón đạm lúc này. Nếu mưa lớn sau phun 6h thì phun nhắc lại."
                }
            ],
            "next_key_stage": "Theo dõi trổ đều → chuẩn bị phun phòng lem lép hạt"
        }
    },

    {
        "farmer_id": "FARMER008",
        "full_name": "Mai Thị Hồng Loan",
        "farm_name": "Ruộng Hồng Phước",
        "area_ha": 1.2,
        "planting_date": "2025-10-10",
        "days_since_planting": 23,
        "rice_variety": "VNR20",
        "location": {"province": "Bạc Liêu"},
        "iot_data": {
            "temperature": 29.3, "humidity": 85, "soil_moisture": 74, "soil_ph": 5.4,
            "water_level": 16, "lux": 39500, "wind": 2.1, "wind_avg": 1.8,
            "detected_disease_name": "none", "disease_confidence": 0.15
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 29.0, "humidity": 88, "soil_moisture": 78, "soil_ph": 5.4, "water_level": 18, "lux": 36200, "wind": 1.5, "wind_avg": 1.3},
            "2/11/2025": {"temperature": 29.6, "humidity": 82, "soil_moisture": 71, "soil_ph": 5.5, "water_level": 14, "lux": 42500, "wind": 2.6, "wind_avg": 2.3},
            "3/11/2025": {"temperature": 29.3, "humidity": 85, "soil_moisture": 74, "soil_ph": 5.4, "water_level": 16, "lux": 39500, "wind": 2.1, "wind_avg": 1.8}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-02T00:00:00Z",
            "stage_name": "Bón Đón Đẻ - Thúc Nhánh",
            "main_message": "CẦN BÓN ĐÓN ĐẺ NGAY NGÀY 2/11/2025 vì cây 23 ngày đang đẻ nhánh đợt 1, pH đất thấp (5.4), thời tiết 2/11 rất thuận lợi.",
            "analysis": {
                "nutrient_need_assessment": "Cây giống VNR20 phát triển nhanh, đang đẻ nhánh đợt đầu, đất chua → cần bón lân + đạm cao để tăng sức đề kháng và số nhánh.",
                "optimal_timing_summary": "Ngày 2/11 mực nước 14cm, trời râm, gió nhẹ → lý tưởng để bón phân."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "20-25 ngày sau sạ",
                    "objective": "Thúc đẻ nhánh đợt 1 mạnh và đều",
                    "key_indicators": "Cây cao 25-30cm, bắt đầu đẻ nhánh, lá hơi vàng do đất chua.",
                    "fertilizers": [
                        {
                            "type": "NPK 20-20-10 + TE",
                            "recommended_dosage_per_unit": "200-250 kg/ha",
                            "calculation_details": "1.2 ha × 230 kg/ha = 276 kg",
                            "quantity_kg": 276.0,
                            "instructions": "Bón vãi đều sáng sớm, giữ nước 4-6cm sau bón."
                        }
                    ],
                    "important_notes": "Có thể bổ sung 20kg vôi bột/ha nếu pH vẫn dưới 5.3 sau 7 ngày."
                }
            ],
            "next_key_stage": "Bón thúc đẻ nhánh đợt 2 (30-35 ngày)"
        }
    },

    {
        "farmer_id": "FARMER009",
        "full_name": "Đỗ Văn Thành",
        "farm_name": "Ruộng Thành Đạt",
        "area_ha": 2.8,
        "planting_date": "2025-08-28",
        "days_since_planting": 66,
        "rice_variety": "OM380",
        "location": {"province": "Hậu Giang"},
        "iot_data": {
            "temperature": 30.8, "humidity": 76, "soil_moisture": 55, "soil_ph": 6.3,
            "water_level": 5, "lux": 65800, "wind": 4.3, "wind_avg": 3.9,
            "detected_disease_name": "none", "disease_confidence": 0.11
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 30.5, "humidity": 79, "soil_moisture": 59, "soil_ph": 6.3, "water_level": 8, "lux": 59800, "wind": 3.5, "wind_avg": 3.1},
            "2/11/2025": {"temperature": 31.2, "humidity": 73, "soil_moisture": 52, "soil_ph": 6.4, "water_level": 4, "lux": 69800, "wind": 4.8, "wind_avg": 4.3},
            "3/11/2025": {"temperature": 30.8, "humidity": 76, "soil_moisture": 55, "soil_ph": 6.3, "water_level": 5, "lux": 65800, "wind": 4.3, "wind_avg": 3.9}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-03T00:00:00Z",
            "stage_name": "Phun bón lá nuôi hạt + giữ nước vào chắc",
            "main_message": "CẦN PHUN BÓN LÁ CAO KALI + VI LƯỢNG NGÀY 3/11/2025 vì cây 66 ngày đang vào chắc xanh, thời tiết nắng nóng, ruộng hơi khô.",
            "analysis": {
                "nutrient_need_assessment": "Hạt đã vào chắc xanh 60-70%, lá đốt cuối còn xanh nhưng có dấu hiệu thiếu Kali nhẹ.",
                "optimal_timing_summary": "Ngày 3/11 nắng đều, gió mạnh vừa → thuốc bám tốt khi phun buổi sáng."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "60-70 ngày sau sạ (vào chắc xanh)",
                    "objective": "Tăng tỷ lệ hạt chắc, nâng khối lượng nghìn hạt",
                    "key_indicators": "Hạt trong suốt → đục sữa, lá cờ còn xanh.",
                    "fertilizers": [
                        {
                            "type": "Phân bón lá 10-10-40 + TE (cao Kali)",
                            "recommended_dosage_per_unit": "2-2.5 kg/ha, pha 600 lần",
                            "calculation_details": "2.8 ha × 2.2 kg/ha = 6.16 kg (làm tròn 6.2 kg)",
                            "quantity_kg": 6.2,
                            "instructions": "Phun đều 2 mặt lá vào 7-9h sáng 3/11, phun lần 2 sau 7 ngày."
                        }
                    ],
                    "important_notes": "Giữ nước 3-5cm liên tục đến thu hoạch."
                }
            ],
            "next_key_stage": "Theo dõi chín → cắt nước (dự kiến 80-85 ngày)"
        }
    },

    {
        "farmer_id": "FARMER010",
        "full_name": "Bùi Thị Kim Cuc",
        "farm_name": "Ruộng Cuc Hương",
        "area_ha": 4.5,
        "planting_date": "2025-09-18",
        "days_since_planting": 45,
        "rice_variety": "HT1",
        "location": {"province": "Vĩnh Long"},
        "iot_data": {
            "temperature": 28.9, "humidity": 89, "soil_moisture": 73, "soil_ph": 5.6,
            "water_level": 13, "lux": 38500, "wind": 1.9, "wind_avg": 1.6,
            "detected_disease_name": "bacterial_leaf_blight", "disease_confidence": 0.91
        },
        "summary_3d": {
            "1/11/2025": {"temperature": 28.6, "humidity": 92, "soil_moisture": 77, "soil_ph": 5.6, "water_level": 16, "lux": 34800, "wind": 1.3, "wind_avg": 1.1},
            "2/11/2025": {"temperature": 29.2, "humidity": 86, "soil_moisture": 70, "soil_ph": 5.7, "water_level": 11, "lux": 41500, "wind": 2.4, "wind_avg": 2.1},
            "3/11/2025": {"temperature": 28.9, "humidity": 89, "soil_moisture": 73, "soil_ph": 5.6, "water_level": 13, "lux": 38500, "wind": 1.9, "wind_avg": 1.6}
        },
        "expect_plan": {
            "is_action_needed": True,
            "execution_date": "2025-11-01T00:00:00Z",
            "stage_name": "Phun thuốc trị bạc lá vi khuẩn + bón thúc 2",
            "main_message": "KHẨN: PHUN TRỊ BẠC LÁ VI KHUẨN + BÓN THÚC 2 NGAY 1/11/2025 vì bệnh bạc lá đang bùng phát mạnh (91%) ở giai đoạn cuối làm đòng.",
            "analysis": {
                "nutrient_need_assessment": "Bệnh bạc lá vi khuẩn lan rộng do thừa nước và đạm, cần kết hợp trị bệnh + bón NPK cân đối.",
                "optimal_timing_summary": "Ngày 1/11 độ ẩm cực cao, gió nhẹ → thuốc trị vi khuẩn phát huy tối đa."
            },
            "fertilizer_stage_detail": [
                {
                    "timing": "40-45 ngày sau sạ",
                    "objective": "Trị bạc lá, thúc làm đòng, tăng sức chống chịu",
                    "key_indicators": "Vết bạc lá lan từ mé hoja đến giữa lá, mép lá khô cháy.",
                    "fertilizers": [
                        {
                            "type": "Copper hydroxide + Kasugamycin",
                            "recommended_dosage_per_unit": "1.0-1.2 kg/ha",
                            "calculation_details": "4.5 ha × 1.1 kg/ha = 4.95 kg (làm tròn 5.0 kg)",
                            "quantity_kg": 5.0,
                            "instructions": "Phun kỹ 2 mặt lá vào chiều 1/11, phun lần 2 sau 5 ngày."
                        },
                        {
                            "type": "NPK 16-16-16 + TE",
                            "recommended_dosage_per_unit": "180-200 kg/ha",
                            "calculation_details": "4.5 ha × 190 kg/ha = 855 kg",
                            "quantity_kg": 855.0,
                            "instructions": "Bón sau phun thuốc 2-3 ngày, giữ nước 5-7cm."
                        }
                    ],
                    "important_notes": "Thoát nước 1 ngày trước phun thuốc, sau đó bón phân."
                }
            ],
            "next_key_stage": "Theo dõi trổ bông (55-65 ngày)"
        }
    },
    {
    "farmer_id": "FARMER012",
    "full_name": "Nguyễn Thị Kim Ngân",
    "farm_name": "Ruộng Ngân Hà",
    "area_ha": 3.0,
    "planting_date": "2025-09-12",
    "days_since_planting": 51,
    "rice_variety": "OM 18",
    "location": {
        "province": "An Giang"
    },
    "iot_data": {
        "temperature": 29.2,
        "humidity": 86,
        "soil_moisture": 68,
        "soil_ph": 5.7,
        "water_level": 12,
        "lux": 45800,
        "wind": 2.6,
        "wind_avg": 2.3,
        "detected_disease_name": "dirty_panicle",
        "disease_confidence": 0.88
    },
    "summary_3d": {
        "1/11/2025": {"temperature": 28.9, "humidity": 89, "soil_moisture": 72, "soil_ph": 5.7, "water_level": 15, "lux": 41200, "wind": 1.8, "wind_avg": 1.6},
        "2/11/2025": {"temperature": 29.5, "humidity": 83, "soil_moisture": 65, "soil_ph": 5.8, "water_level": 10, "lux": 49200, "wind": 3.1, "wind_avg": 2.8},
        "3/11/2025": {"temperature": 29.2, "humidity": 86, "soil_moisture": 68, "soil_ph": 5.7, "water_level": 12, "lux": 45800, "wind": 2.6, "wind_avg": 2.3}
    },
    "expect_plan": {
        "is_action_needed": True,
        "execution_date": "2025-11-01T00:00:00Z",
        "stage_name": "Phun phòng lem lép hạt (bông đen) khẩn cấp",
        "main_message": "KHẨN CẤP: PHUN PHÒNG LEM LÉP HẠT NGAY CHIỀU 1/11/2025 vì cây đã trổ lác đác 3 ngày nay, phát hiện bệnh lem lép hạt (dirty panicle) mức 88%, trời ngày 1/11 rất thuận lợi để phun thuốc.",
        "analysis": {
            "nutrient_need_assessment": "Cây OM18 51 ngày đang trổ lác đác → trổ đều trong 3-5 ngày tới. Bệnh lem lép hạt do nấm và vi khuẩn đã xuất hiện mạnh do độ ẩm cao liên tục và dư đạm từ đợt trước.",
            "optimal_timing_summary": "Ngày 1/11 có mây nhiều, gió nhẹ, độ ẩm cao, không mưa → điều kiện hoàn hảo để thuốc trừ nấm + vi khuẩn bám đều và phát huy tác dụng lâu."
        },
        "fertilizer_stage_detail": [
            {
                "timing": "Trổ lác đác (bắt đầu 5-10% bông)",
                "objective": "Phòng trị lem lép hạt, bảo vệ hạt khỏi nấm và vi khuẩn gây đen bông, giảm tỷ lệ hạt lép",
                "key_indicators": "Bông vừa nhú khỏi bẹ lá đòn, hạt phấn hoa trắng, phát hiện hạt bị đen/nâu trên bông sớm.",
                "fertilizers": [
                    {
                        "type": "Tilt Super 300EC (Propiconazole + Difenoconazole) hoặc Amistar Top + Kasumin",
                        "recommended_dosage_per_unit": "0.5-0.6 lít/ha",
                        "calculation_details": "3.0 ha × 0.55 lít/ha = 1.65 lít (làm tròn 1.7 lít)",
                        "quantity_kg": 1.7,
                        "instructions": "Phun ướt đều bông và lá đòn vào 15h-17h ngày 1/11/2025. Phun lần 2 khi trổ đều (dự kiến 5-7/11)."
                    },
                    {
                        "type": "Phân bón lá cao Kali + Canxi Bo (chống lem lép sinh lý)",
                        "recommended_dosage_per_unit": "1.5-2 kg/ha, pha 800 lần",
                        "calculation_details": "3.0 ha × 1.8 kg/ha = 5.4 kg",
                        "quantity_kg": 5.4,
                        "instructions": "Phun chung đợt với thuốc BVTV để tăng sức chống chịu."
                    }
                ],
                "important_notes": "Giữ nước nông 3-5cm, tuyệt đối không để ngập sâu. Nếu mưa lớn sau phun 4 tiếng thì phun nhắc lại liều 50% sau 3 ngày."
            }
        ],
        "next_key_stage": "Phun lần 2 khi trổ đều (80-90% bông) → theo dõi vào chắc hạt"
    }
}
]

In [63]:
def _get_store_paths(store_name: str):
    """Tạo đường dẫn file động cho một kho tri thức cụ thể."""
    base_dir = r"D:\finalproject\KLTN\Backend\data\vector_store"
    index_path = os.path.join(base_dir, f"faiss_index_{store_name}.bin")
    docs_path = os.path.join(base_dir, f"documents_{store_name}.json")
    return index_path, docs_path

_stores = {}

def get_store(store_name: str):
    """
    Lấy một kho tri thức cụ thể. Tải từ cache nếu có, nếu không thì xây dựng mới.
    """
    if store_name in _stores:
        return _stores[store_name]

    index_path, docs_path = _get_store_paths(store_name)

    if os.path.exists(index_path) and os.path.exists(docs_path):
        try:
            print(f"Đang tải kho '{store_name}' từ cache...")
            index = faiss.read_index(index_path)
            with open(docs_path, 'r', encoding='utf-8') as f:
                documents = json.load(f)
            print(f"Tải thành công kho '{store_name}' với {index.ntotal} vector.")
            
            store_instance = {"index": index, "documents": documents}
            _stores[store_name] = store_instance
            return store_instance
        except Exception as e:
            print(f"Lỗi khi tải kho '{store_name}' từ cache: {e}. Sẽ xây dựng lại.")

def retrieve(store_name: str, query: str, k: int = 5) -> str:
    """Thực hiện truy vấn trên một kho tri thức chuyên biệt."""
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
    store = get_store(store_name)
    if not store or store.get("index") is None:
        print(f"Truy vấn thất bại: Kho tri thức '{store_name}' chưa được khởi tạo.")
        return f"Lỗi: Cơ sở tri thức '{store_name}' không khả dụng."

    index = store["index"]
    documents = store["documents"]
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    
    try:
        _, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
        retrieved_docs = [documents[i] for i in indices[0]]
        context = "\n---\n".join([doc['content'] for doc in retrieved_docs])
        
        print(f"Đã truy xuất {len(retrieved_docs)} đoạn văn bản từ kho '{store_name}' cho câu hỏi: '{query[:50]}...'")
        return context
    except Exception as e:
        print(f"Lỗi trong quá trình truy xuất từ kho '{store_name}': {e}")
        return "Lỗi: Đã xảy ra sự cố khi tìm kiếm thông tin."

In [64]:
from sentence_transformers import SentenceTransformer, util

eval_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

def text_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    emb = eval_model.encode([a, b], convert_to_tensor=True)
    return float(util.cos_sim(emb[0], emb[1]).item())


def determine_current_stage(days: int) -> str:
    if days < 0:
        return "Bón Lót/Chưa sạ"
    elif days <= 10:
        return "Bón Thúc 1 - Phục hồi và Đẻ nhánh sớm"
    elif days <= 25:
        return "Bón Thúc 1 - Đẻ Nhánh Rộ/Dưỡng lá"
    elif days <= 34:
        return "Giai đoạn Đệm - Dưỡng lá và Kiểm soát chồi vô hiệu"
    elif days <= 45:
        return "Bón Thúc 2 - Làm Đòng/Đón Đòng"
    elif days <= 59:
        return "Giai đoạn Trổ Bông/Trổ Xong (Không bón chính)"
    elif days <= 80:
        return "Bón Nuôi Hạt (Nếu cần bón lá để tăng hạt chắc)"
    else:
        return "Giai đoạn Chờ Thu hoạch"
    
def select_best_execution_date(summary_3d: dict) -> str:
    scores = {}
    for date_str, data in summary_3d.items():
        score = 100
        # Ưu tiên mực nước lý tưởng 10-15cm
        water = data.get("water_level", 0)
        if 10 <= water <= 15:
            score += 20
        if water < 8 or water > 20:
            score -= 30
        # Ưu tiên độ ẩm đất cao
        if data.get("soil_moisture", 0) >= 68:
            score += 15
        # Ưu tiên gió nhẹ + trời râm
        if data.get("wind_avg", 99) <= 2.0:
            score += 10
        if data.get("lux", 99999) <= 45000:  # trời nhiều mây
            score += 10
        scores[date_str] = score
    
    best_date_str = max(scores, key=scores.get)
    # Chuyển "2/11/2025" → "2025-11-02T00:00:00Z" (7h sáng VN = 00:00 UTC)
    dt = datetime.strptime(best_date_str, "%d/%m/%Y")
    execution_iso = dt.strftime("%Y-%m-%dT00:00:00Z")
    return execution_iso, best_date_str

In [65]:
import json
from datetime import datetime, timezone
from typing import Dict, List, Optional

def build_nutrient_prompt(
    retrieved_context: str,
    farmer_info: dict,
    summary_3d: dict,
    iot_data: dict,
    current_stage: str,
    best_execution_date: str
) -> str:
    FORCE_DATE = best_execution_date
    current_utc = datetime.now(timezone.utc).isoformat(timespec='seconds')[:-6] + 'Z'
    
    farmer_json = json.dumps(farmer_info, ensure_ascii=False, indent=2)
    summary_json = json.dumps(summary_3d, ensure_ascii=False, indent=2)
    iot_json = json.dumps(iot_data, ensure_ascii=False, indent=2)

    return f"""
    Bạn là chuyên gia nông học lúa nước hàng đầu Việt Nam. Thời gian hiện tại (UTC): {current_utc}

    **NHIỆM VỤ DUY NHẤT:** Tạo kế hoạch bón phân CHO GIAI ĐOẠN HIỆN TẠI dựa trên dữ liệu thực tế.

    **THÔNG TIN ĐẦU VÀO:**
    1. Giai đoạn hiện tại: {current_stage}
    2. Kiến thức nền (trích dẫn bắt buộc khi dùng):
    {retrieved_context}

    3. Thông tin nông hộ:
    {farmer_json}

    4. IoT hiện tại (ruộng đang rất khô + đạo ôn nặng):
    {iot_json}

    5. Dự báo 3 ngày tới (ngày tốt nhất đã được chọn: {best_execution_date[:10]}):
    {summary_json}

    === YÊU CẦU KHÔNG ĐƯỢC PHÁ VỠ (nếu vi phạm sẽ bị 0 điểm):
    • execution_date PHẢI LÀ: "{FORCE_DATE}" (không được chọn ngày khác)
    • main_message phải bắt đầu bằng "CẦN BÓN THÚC 2 NGAY" hoặc "CẬP NHẬT KẾ HOẠCH BÓN"
    • Phải đề cập rõ bệnh đạo ôn nặng → tăng Kali + Silic
    • Liều lượng tính chính xác cho {farmer_info['area_ha']} ha
    • Trả về ĐÚNG 100% định dạng JSON dưới đây, không thêm bất kỳ từ nào khác

    **JSON ĐẦU RA (EXACT):**
    ```json
                {{
                    "is_action_needed": "boolean (True nếu cần bón/cập nhật, False nếu đã bón đủ)",
                    "execution_date": "YYYY-MM-DDTHH:MM:SSZ (BẮT BUỘC. Thời điểm bắt đầu thực thi, múi giờ UTC, ví dụ: '2025-11-05T07:00:00Z')",
                    "stage_name": "string (Tên giai đoạn: {current_stage})",
                    "main_message": "string (Tóm tắt hành động. Ghi rõ nếu là CẬP NHẬT, BÓN BỔ SUNG, hay KHÔNG CẦN BÓN)",
                    "analysis": {{
                        "nutrient_need_assessment": "string (Phân tích nhu cầu NPK, có so sánh nếu cần)",
                        "optimal_timing_summary": "string (Phân tích ngày/giờ bón tối ưu MỚI NHẤT)"
                    }},
                    "fertilizer_stage_detail": [
                        {{
                            "timing": "string (Khoảng ngày NSS, ví dụ: '7-10 NSS')",
                            "objective": "string",
                            "key_indicators": "string (Dấu hiệu nhận biết, kèm trích dẫn)",
                            "fertilizers": [
                                {{
                                    "type": "string",
                                    "recommended_dosage_per_unit": "string (Trích dẫn liều lượng khuyến nghị, (1))",
                                    "calculation_details": "string (Hiển thị phép tính cho {farmer_info.get("area_ha")} ha)",
                                    "quantity_kg": "float",
                                    "instructions": "string (Hướng dẫn kỹ thuật bón, mực nước)"
                                }}
                            ],
                            "important_notes": "string (Cảnh báo, lưu ý)"
                        }}
                    ],
                    "next_key_stage": "string (Giai đoạn bón phân quan trọng tiếp theo)"
                }}
                ```
    Chỉ trả về JSON hợp lệ, không giải thích.
    """

# ===============================================
# 6. HÀM TẠO KẾ HOẠCH (giống hệt NutrientAgent)
# ===============================================
def create_nutrient_plan(sample_item: dict, client) -> dict:
    data = sample_item.copy()
    expect_plan = data.pop("expect_plan", None)  

    days = data["days_since_planting"]
    current_stage = determine_current_stage(days)
    query_for_retrieval = f"Công thức và liều lượng bón phân chi tiết giai đoạn {current_stage} cho lúa {data.get('rice_variety')}..."
    retrieved_context = retrieve("fertilizer_management", query_for_retrieval, k=6)
    best_date_iso, best_date_vn = select_best_execution_date(data["summary_3d"])

    # Build prompt giống hệt NutrientAgent
    prompt = build_nutrient_prompt(
        retrieved_context=retrieved_context,
        farmer_info=data,
        summary_3d=data["summary_3d"],
        iot_data=data["iot_data"],
        current_stage=current_stage,
        best_execution_date=best_date_iso
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        generated_plan = json.loads(response.choices[0].message.content)
        return {
            "generated_plan": generated_plan,
            "expect_plan": expect_plan,
            "match_execution_date": generated_plan.get("execution_date") == expect_plan.get("execution_date"),
            "stage_correct": generated_plan.get("stage_name") == current_stage
        }
    except Exception as e:
        return {"error": str(e)}

In [66]:
import time
from statistics import mean, stdev

def evaluate_nutrient_agent(test_data: List[dict], client) -> dict:
    results = []
    latencies = []

    print(f"Bắt đầu đánh giá trên {len(test_data)} mẫu...\n")

    for i, item in enumerate(test_data):
        print(f"\n=== Test case {i+1}/{len(test_data)} | Farmer: {item['full_name']} ===")
        
        start = time.time()
        result = create_nutrient_plan(item, client)
        latency = time.time() - start
        latencies.append(latency)

        # Debug log
        if "" in result:
            print(result.get(""))

        # Nếu lỗi → bỏ qua sample
        if "error" in result:
            print("❌ ERROR:", result["error"])
            continue

        gen = result["generated_plan"]
        exp = result["expect_plan"]

        # Similarity metrics
        msg_sim = text_similarity(
            gen.get("main_message", ""), 
            exp.get("main_message", "")
        )
        ana_sim = text_similarity(
            gen.get("analysis", {}).get("nutrient_need_assessment", ""),
            exp.get("analysis", {}).get("nutrient_need_assessment", "")
        )

        case_result = {
            "test_id": i,
            "farmer_name": item['full_name'],
            "execution_date_match": result["match_execution_date"],
            "stage_correct": result["stage_correct"],
            "main_message_similarity": round(msg_sim, 3),
            "analysis_similarity": round(ana_sim, 3),
            "overall_score": round(
                (msg_sim + ana_sim +
                 (1 if result["match_execution_date"] else 0) +
                 (1 if result["stage_correct"] else 0)) / 4,
                3
            ),
            "latency": round(latency, 3)
        }

        results.append(case_result)

        print(
            f"  ✔ Date match: {case_result['execution_date_match']} | "
            f"Stage correct: {case_result['stage_correct']} | "
            f"Msg sim: {case_result['main_message_similarity']} | "
            f"Ana sim: {case_result['analysis_similarity']} | "
            f"Latency: {case_result['latency']}s | "
            f"Score: {case_result['overall_score']}"
        )

    # ===========================
    #   SUMMARY METRICS
    # ===========================

    avg_score = mean(r["overall_score"] for r in results)
    avg_msg_sim = mean(r["main_message_similarity"] for r in results)
    avg_ana_sim = mean(r["analysis_similarity"] for r in results)
    stage_accuracy = mean(1 if r["stage_correct"] else 0 for r in results)
    date_match_rate = mean(1 if r["execution_date_match"] else 0 for r in results)
    
    avg_latency = mean(latencies)
    std_latency = stdev(latencies) if len(latencies) > 1 else 0

    print("\n==================== TỔNG KẾT ====================")
    print(f"🧪 Tổng số case: {len(results)}")
    print(f"⚡ Latency trung bình: {avg_latency:.2f}s (std = {std_latency:.2f})")
    print(f"🎯 Độ chính xác stage: {stage_accuracy:.2f}")
    print(f"📅 Độ khớp ngày bón: {date_match_rate:.2f}")
    print(f"📝 Similarity main message TB: {avg_msg_sim:.3f}")
    print(f"📊 Similarity analysis TB: {avg_ana_sim:.3f}")
    print(f"⭐ Điểm TB tổng hợp: {avg_score:.3f}")
    print("==================================================\n")

    return {
        "detailed": results,
        "summary": {
            "average_score": round(avg_score, 3),
            "average_main_msg_similarity": round(avg_msg_sim, 3),
            "average_analysis_similarity": round(avg_ana_sim, 3),
            "stage_accuracy": round(stage_accuracy, 3),
            "date_match_rate": round(date_match_rate, 3),
            "average_latency": round(avg_latency, 3),
            "latency_std": round(std_latency, 3),
            "total_cases": len(results)
        }
    }


In [67]:
evaluation_result = evaluate_nutrient_agent(sample_farmer_data, client)

Bắt đầu đánh giá trên 10 mẫu...


=== Test case 1/10 | Farmer: Nguyễn Văn An ===
Đang tải kho 'fertilizer_management' từ cache...
Tải thành công kho 'fertilizer_management' với 184 vector.
Đã truy xuất 6 đoạn văn bản từ kho 'fertilizer_management' cho câu hỏi: 'Công thức và liều lượng bón phân chi tiết giai đoạ...'
  ✔ Date match: True | Stage correct: True | Msg sim: 0.358 | Ana sim: 0.663 | Latency: 18.529s | Score: 0.755

=== Test case 2/10 | Farmer: Trần Thị Bé ===
Đã truy xuất 6 đoạn văn bản từ kho 'fertilizer_management' cho câu hỏi: 'Công thức và liều lượng bón phân chi tiết giai đoạ...'
  ✔ Date match: False | Stage correct: True | Msg sim: 0.284 | Ana sim: 0.669 | Latency: 13.992s | Score: 0.488

=== Test case 3/10 | Farmer: Lê Văn Hùng ===
Đã truy xuất 6 đoạn văn bản từ kho 'fertilizer_management' cho câu hỏi: 'Công thức và liều lượng bón phân chi tiết giai đoạ...'
  ✔ Date match: False | Stage correct: True | Msg sim: 0.47 | Ana sim: 0.555 | Latency: 17.897s | Score: 0.506



In [68]:
s = evaluation_result["summary"]
print(
    f"Score TB: {s['average_score']} | "
    f"MsgSim: {s['average_main_msg_similarity']} | "
    f"AnaSim: {s['average_analysis_similarity']} | "
    f"StageAcc: {s['stage_accuracy']} | "
    f"DateMatch: {s['date_match_rate']} | "
    f"LatencyTB: {s['average_latency']}s"
)


Score TB: 0.621 | MsgSim: 0.361 | AnaSim: 0.523 | StageAcc: 1 | DateMatch: 0.6 | LatencyTB: 16.661s
